# Student Performance - ML Pipeline
Predicting **pass/fail** (classification) and **final score** (regression)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob, os, warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, RocCurveDisplay,
                             ConfusionMatrixDisplay, classification_report,
                             mean_squared_error, r2_score)

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

csv_path = glob.glob('/kaggle/input/**/*.csv', recursive=True)[0]
df = pd.read_csv(csv_path)
df.head()

## 1. Preprocessing

In [ ]:
# Encode categoricals
le = LabelEncoder()
for col in ['gender', 'parent_education', 'internet_access', 'extracurricular']:
    df[col] = le.fit_transform(df[col])

df['passed'] = df['passed'].map({'Yes': 1, 'No': 0})

features = ['gender', 'age', 'study_hours_per_week', 'attendance_rate',
            'parent_education', 'internet_access', 'extracurricular', 'previous_score']

X = df[features]
y_clf = df['passed']
y_reg = df['final_score']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_scaled, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_scaled, y_reg, test_size=0.2, random_state=42)

print(f'Train: {X_train_c.shape[0]}  Test: {X_test_c.shape[0]}')
print(f'Features: {features}')
print(f'Pass rate: {y_clf.mean():.1%}')

## 2. Classification - Predict Pass/Fail

In [ ]:
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'SVM (RBF)': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7)
}

clf_results = []
for name, model in clf_models.items():
    model.fit(X_train_c, y_train_c)
    y_pred = model.predict(X_test_c)
    acc = accuracy_score(y_test_c, y_pred)
    f1 = f1_score(y_test_c, y_pred)
    try:
        auc = roc_auc_score(y_test_c, model.predict_proba(X_test_c)[:, 1])
    except:
        auc = 0
    clf_results.append({'Model': name, 'Accuracy': acc, 'F1': f1, 'AUC': auc})

clf_df = pd.DataFrame(clf_results).sort_values('AUC', ascending=False)
clf_df.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar comparison
x = np.arange(len(clf_df))
w = 0.25
axes[0].bar(x - w, clf_df['Accuracy'], w, label='Accuracy', color='#3b82f6')
axes[0].bar(x, clf_df['F1'], w, label='F1', color='#f97316')
axes[0].bar(x + w, clf_df['AUC'], w, label='AUC', color='#10b981')
axes[0].set_xticks(x)
axes[0].set_xticklabels(clf_df['Model'], rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('Score')
axes[0].set_title('Classification Model Comparison', fontweight='bold', color='white')
axes[0].legend()
axes[0].spines[['top', 'right']].set_visible(False)

# Best model confusion matrix
best_clf_name = clf_df.iloc[0]['Model']
best_clf = clf_models[best_clf_name]
ConfusionMatrixDisplay.from_estimator(best_clf, X_test_c, y_test_c,
    cmap='Blues', ax=axes[1], colorbar=False)
axes[1].set_title(f'Confusion Matrix - {best_clf_name}', fontweight='bold', color='white')

for ax in axes:
    ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#3b82f6', '#f97316', '#10b981', '#a855f7', '#ef4444']
for (name, model), color in zip(clf_models.items(), colors):
    RocCurveDisplay.from_estimator(model, X_test_c, y_test_c, ax=ax, name=name, color=color)

ax.plot([0, 1], [0, 1], 'w--', alpha=0.3)
ax.set_title('ROC Curves - All Models', fontweight='bold', color='white')
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from best tree model
tree_clf = clf_models['Gradient Boosting']
imp = pd.Series(tree_clf.feature_importances_, index=features).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
imp.plot.barh(ax=ax, color='#f97316', edgecolor='none')
ax.set_title('Feature Importance (Classification - Gradient Boosting)', fontweight='bold', color='white')
ax.set_xlabel('Importance')
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 3. Regression - Predict Final Score

In [ ]:
reg_models = {
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42)
}

reg_results = []
predictions = {}
for name, model in reg_models.items():
    model.fit(X_train_r, y_train_r)
    y_pred = model.predict(X_test_r)
    predictions[name] = y_pred
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred))
    r2 = r2_score(y_test_r, y_pred)
    cv = cross_val_score(model, X_scaled, y_reg, cv=5, scoring='r2').mean()
    reg_results.append({'Model': name, 'RMSE': rmse, 'R2 (Test)': r2, 'R2 (CV)': cv})

reg_df = pd.DataFrame(reg_results).sort_values('R2 (Test)', ascending=False)
reg_df.round(3)

In [ ]:
best_reg_name = reg_df.iloc[0]['Model']
best_pred = predictions[best_reg_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test_r, best_pred, alpha=0.5, s=25, color='#3b82f6', edgecolors='#333', linewidth=0.3)
lo, hi = min(y_test_r.min(), best_pred.min()), max(y_test_r.max(), best_pred.max())
axes[0].plot([lo, hi], [lo, hi], 'w--', alpha=0.5, label='Perfect')
axes[0].set_xlabel('Actual Score')
axes[0].set_ylabel('Predicted Score')
axes[0].set_title(f'Actual vs Predicted ({best_reg_name})', fontweight='bold', color='white')
axes[0].legend()
axes[0].spines[['top', 'right']].set_visible(False)

# Residual distribution
residuals = y_test_r.values - best_pred
axes[1].hist(residuals, bins=30, color='#a855f7', edgecolor='#1a1a2e', alpha=0.9)
axes[1].axvline(0, color='#f97316', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', fontweight='bold', color='white')
axes[1].spines[['top', 'right']].set_visible(False)

for ax in axes:
    ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for regression
tree_reg = reg_models['Gradient Boosting']
imp_r = pd.Series(tree_reg.feature_importances_, index=features).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
imp_r.plot.barh(ax=ax, color='#10b981', edgecolor='none')
ax.set_title('Feature Importance (Regression - Gradient Boosting)', fontweight='bold', color='white')
ax.set_xlabel('Importance')
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()